# Implementando o EEGNet em Haskell

Após desenvolver o **Lambda Ai**, nosso framework de aprendizagem profunda, unimos a pesquisa de Mateus a respeito da arquitetura mais simples para o EEGNet para replicar nesta apresentação.

Nosso maior foco foi trazer de forma compacta em um jupyter server a inferência do modelo de aprendizagem de máquina.

---

## Importando os pacotes necessários

Esta seção vai importar os pacotes necessários que precisamos de nosso framework para treinamento

In [1]:
:set -i/opt/library/

In [2]:
import Layers (MetaLayer(..), genLayer')
import Convolutional (ConvolutionalLayer'(..))
import ActivationFunction (ActivationFunction'(..))
import Model (Model(Model), train, forward)
import Utils (readCSV)
import LossFunctions (mseLF)
import Architeture (Inputs, Targets, Dataset, ResultsAll)

In [3]:
import Data.List (maximumBy)
import Data.Ord (comparing)

In [4]:
import GHC.IO (unsafePerformIO)

In [5]:
import GHC.Conc (getNumProcessors, getNumCapabilities, setNumCapabilities)

do
  setNumCapabilities 1
  procs <- getNumProcessors
  caps  <- getNumCapabilities
  putStrLn $ "CPUs detectadas pelo SO/Docker: " ++ show procs
  putStrLn $ "Threads em uso no IHaskell:    " ++ show caps

CPUs detectadas pelo SO/Docker: 4
Threads em uso no IHaskell:    1

---

## Configurações

Aqui iremos estabelecer as constantes e outras configurações para fácil legibilidade

Caminhos para os datasets

In [6]:
dataTrain      = "data/train.csv"
dataTest       = "data/test.csv"
dataValidation = "data/validation.csv"

Características de nossa base

In [7]:
numeroClasses = 4
numeroCanais  = 22
numeroFiltros = 40
tamanhoKernel = 25

In [8]:
primeiraDimensao = 22
segundaDimensao  = 141
dimensoesEntrada = [primeiraDimensao, segundaDimensao]

In [9]:
quantidadeEpocas  = 2000
quantidadeBatches = 1

Resultados da inferência

In [10]:
arquivoInferencia = "data/predicao_definitiva.csv"

---

## Carregando o Data Set

Carregamento tanto do data set de treino como de teste

In [11]:
getLabel :: [Double] -> Double
getLabel = last

In [12]:
tiraLabel :: [Double] -> [Double]
tiraLabel = init

In [13]:
paraOneHot :: Double -> [Double]
paraOneHot i = [ if i==j then 1 else 0 | j <- [0..pred numeroClasses]]

In [14]:
preProcessaLinha :: [Double] -> (Inputs, Targets)
preProcessaLinha linha = ((tiraLabel linha, dimensoesEntrada), ((paraOneHot . getLabel) linha, [4, 1, 1, 1, 1]))

In [15]:
generateDataset :: String -> IO Dataset
generateDataset datasetPath = do
    csv <- readCSV datasetPath
    return $ (filter ( (>0) . sum. fst. snd) . map preProcessaLinha . take 10) csv

In [16]:
trainSet       = (unsafePerformIO .  generateDataset) dataTrain

In [17]:
testSet        = (unsafePerformIO . generateDataset) dataTest

In [18]:
validationSet  = (unsafePerformIO . generateDataset) dataValidation

---

## Definindo arquitetura de meu modelo

Utilizando apenas as camadas convolucionais e funções de ativação, vamos definir nossa arquitetura

In [19]:
eegMetaLayers :: [MetaLayer]
eegMetaLayers = 
    [
         MetaConvolution (ConvolutionalLayer' { learnRate' = 0.05, filters = numeroFiltros, dimensions = [1, tamanhoKernel] }), 
         -- (c,s)x(1,t)=(c,s-t+1)=>(f,c,s-t+1)
         MetaConvolution (ConvolutionalLayer' { learnRate' = 0.05, filters = numeroFiltros, dimensions = [1, numeroCanais, 1] }),
         -- (f,c,s-t+1)x(1,c,1)=(f,1,s-t+1)=>(f,f,1,s-t+1)
         MetaActivation (ActivationFunction' "leakyRelu"),
         MetaConvolution (ConvolutionalLayer' { learnRate' = 0.05, filters = numeroFiltros, dimensions = [1, 1, 1, 3] }),
         -- (f,f,1,s-t+1)x(1,1,1,3)=(f,f,1,s-t-1)=>(f,f,f,1,s-t-1)
         MetaConvolution (ConvolutionalLayer' { learnRate' = 0.05, filters = numeroClasses, dimensions = [numeroFiltros, numeroFiltros, 1, segundaDimensao - tamanhoKernel + 1] }),
         -- (f,f,f,1,s-t-1)x(f,f,f,1,s-t-1)=(1,1,1,1,1)=>(cls,1,1,1,1,1)
         MetaActivation (ActivationFunction' "sigmoid")
    ]

In [20]:
eegModel = Model <$> mapM genLayer' eegMetaLayers

---

## Treinando o modelo

A partir da arquitetura já definida, iremos treinar o modelo com o conjunto de treinamento

In [21]:
modeloTreinado =
    unsafePerformIO $ do
        modeloInicial <- eegModel 
        return $ train modeloInicial trainSet mseLF (quantidadeEpocas, quantidadeBatches)

Line 1: Missing NOINLINE pragma
Found:
modeloTreinado
  = unsafePerformIO
      $ do modeloInicial <- eegModel
           return
             $ train
                 modeloInicial trainSet mseLF (quantidadeEpocas, quantidadeBatches)
Why not:
{-# NOINLINE modeloTreinado #-}
modeloTreinado
  = unsafePerformIO
      $ do modeloInicial <- eegModel
           return
             $ train
                 modeloInicial trainSet mseLF (quantidadeEpocas, quantidadeBatches)

---

## Inferindo com o modelo

A partir do modelo treinado, iremos realizar inferências a partir do modelo já treinado

In [22]:
inferirTestes :: Dataset -> Model -> ResultsAll
inferirTestes testSet' modeloTreinado' =
    map ((last . forward modeloTreinado') . fst) testSet'

---

## Pegando resultados

A partir das inferências, iremos pegar as labels geradas pelo modelo

In [23]:
argMax :: Ord a => [a] -> Int
argMax lista = fst $ maximumBy (comparing snd) (zip [0..] lista)

In [24]:
pegarResultados :: ResultsAll -> [Int]
pegarResultados = map (argMax . fst) 

In [25]:
resultadosTestSet = (pegarResultados . map snd )testSet

In [26]:
resultadosTestes = pegarResultados $ inferirTestes testSet modeloTreinado

---

## Salvando resultados em disco

Finalizando o processo de salvar os resultados no disco

In [27]:
todosResultadosTexto = zipWith (\inferencia label -> show inferencia ++ "," ++ show label) resultadosTestes resultadosTestSet

In [28]:
writeFile arquivoInferencia (unlines todosResultadosTexto)

: 